# Anonymize → RAG Pipeline Demo

End-to-end walkthrough: raw text files → PII anonymization → vector-store ingestion → RAG query.

**What this notebook covers**

1. **Pydantic pipeline config** — define the full pipeline parameters as a typed Pydantic model (and learn how to export it as YAML)
2. **Sample data** — write a few text files with synthetic PII to a temp directory
3. **Anonymize** — run `anonymize_files_flow` to scrub PII, inspect the `AnonymizeManifest`
4. **Ingest** — run `rag_file_ingestion_flow` on the anonymized output
5. **Query** — search the vector store to confirm anonymized chunks are retrieved
6. **YAML equivalent** — see how the same pipeline maps to `workflows.yaml` profiles

---

> **Why Pydantic instead of YAML here?**  
> YAML profiles (in `config/workflows.yaml`) are perfect for repeatable, CLI-driven pipelines.  
> Pydantic objects are better when you're *building* a pipeline in a notebook: you get type hints,
> IDE completion, runtime validation, and you can inspect / transform the config programmatically.
> The two approaches are equivalent — `config.model_dump()` produces the exact dict a YAML profile would.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)
sys.path.append(".")

## 1. Define the pipeline with a Pydantic config

Instead of editing a YAML file, we specify all parameters in a typed `PipelineConfig` object.
This is convenient for notebooks and scripts; the YAML equivalent is shown in [section 6](#6-yaml-equivalent).

In [2]:
import tempfile

from pydantic import BaseModel, Field


class AnonymizeRagPipelineConfig(BaseModel):
    """Full configuration for the anonymize → RAG ingestion pipeline.

    This is the Pydantic equivalent of two `workflow_profiles` entries in
    config/workflows.yaml (one for anonymize_files, one for rag_add_files).
    """

    # ---- Source files ----
    base_dir: str
    pathspecs: list[str] = Field(default_factory=lambda: ["**/*.txt", "**/*.md"])

    # ---- Anonymization ----
    anon_dir: str
    analyzed_fields: list[str] = Field(
        default_factory=lambda: ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "LOCATION", "CREDIT_CARD"]
    )
    faker_seed: int | None = 42  # None = random fakes each run
    save_mapping: bool = False  # True = write .mapping.json sidecars

    # ---- RAG ingestion ----
    retriever_name: str = "default"
    chunk_size: int = 512
    chunker: str = "auto"
    batch_size: int = 10
    force: bool = False  # True = reprocess even unchanged files


# Create temp directories so the demo is self-contained (no config paths needed)
_tmpdir = tempfile.mkdtemp(prefix="anon_rag_demo_")

config = AnonymizeRagPipelineConfig(
    base_dir=f"{_tmpdir}/source",
    anon_dir=f"{_tmpdir}/anonymized",
)

print("Pipeline config:", config)

Pipeline config:
AnonymizeRagPipelineConfig(
    base_dir='/tmp/anon_rag_demo_pmq93tea/source',
    pathspecs=['**/*.txt', '**/*.md'],
    anon_dir='/tmp/anon_rag_demo_pmq93tea/anonymized',
    analyzed_fields=['PERSON', 'EMAIL_ADDRESS', 'PHONE_NUMBER', 'LOCATION', 'CREDIT_CARD'],
    faker_seed=42,
    save_mapping=False,
    retriever_name='default',
    chunk_size=512,
    chunker='auto',
    batch_size=10,
    force=False
)

## 2. Create sample files with synthetic PII

We write a few text files that contain names, emails, and phone numbers so we can verify the anonymization step.

In [3]:

source_dir = Path(config.base_dir)
source_dir.mkdir(parents=True, exist_ok=True)

SAMPLE_FILES = {
    "contract_001.txt": """\
Service Agreement

This agreement is between Alice Johnson (alice.johnson@acme.com, +1-555-123-4567)
and Bob Smith (bob.smith@globex.io, +1-555-987-6543).

Alice Johnson is based in New York. Bob Smith operates from London.
Payment details: credit card 4111 1111 1111 1111, billing address: 123 Main St.
""",
    "support_ticket_42.txt": """\
Support Ticket #42

Customer: Carol White
Email: carol.white@example.org
Phone: (800) 555-0100
Location: San Francisco, CA

Issue: Unable to log into account. Please contact carol.white@example.org for follow-up.
""",
    "meeting_notes.md": """\
# Q2 Planning Meeting

Attendees: David Lee (david@startup.io), Emma Brown (emma.brown@corp.net)

David Lee proposed the new roadmap. Emma Brown will follow up by email at emma.brown@corp.net.
Next meeting in Chicago on 2026-06-15.
""",
}

for filename, content in SAMPLE_FILES.items():
    (source_dir / filename).write_text(content, encoding="utf-8")
    print(f"  Wrote {filename} ({len(content)} chars)")

print(f"\nSource directory: {source_dir}")

Wrote contract_001.txt (304 chars)

Wrote support_ticket_42.txt (213 chars)

Wrote meeting_notes.md (232 chars)

Source directory: /tmp/anon_rag_demo_pmq93tea/source

## 3. Anonymize files

We call `anonymize_files_flow` directly using `run_flow_ephemeral` — the same helper used by the
workflow CLI.  The flow:

- Detects PII via **Presidio** (backed by spaCy NER)
- Replaces each entity with a **Faker**-generated value
- Writes anonymized copies to `config.anon_dir`
- Saves an `anonymize_manifest.json` for incremental re-runs

The anonymization logic is **shared** with `AnonymizationMiddleware`:  
the same `anonymize_text()` function runs here at ETL time, and inside the agent at inference time.

In [4]:
from genai_tk.workflow.prefect.flows.anonymize_flow import anonymize_files_flow
from genai_tk.workflow.prefect.run import run_flow_ephemeral

anon_manifest = run_flow_ephemeral(
    anonymize_files_flow,
    base_dir=config.base_dir,
    output_dir=config.anon_dir,
    pathspecs=config.pathspecs,
    batch_size=config.batch_size,
    force=config.force,
    save_mapping=config.save_mapping,
    analyzed_fields=config.analyzed_fields,
    faker_seed=config.faker_seed,
)

print(f"\nProcessed {len(anon_manifest.entries)} files")
for path, entry in anon_manifest.entries.items():
    print(f"  {entry.output_path}  (hash={entry.source_hash[:8]}...)")

2026-05-12 18:11:25.388 | INFO     | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_files_flow:205 - Found 3 files matching patterns in '/tmp/anon_rag_demo_pmq93tea/source'
2026-05-12 18:11:25.389 | INFO     | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_files_flow:226 - Processing 3 files, skipping 0 unchanged
2026-05-12 18:11:25.390 | INFO     | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_files_flow:234 - Batch 1/1 (3 files)
2026-05-12 18:11:26.654 | INFO     | genai_tk.utils.spacy_model_mngr:setup_spacy_model:103 - SpaCy model 'en_core_web_sm' is available globally


18:11:26.917 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - CreditCardRecognizer supported languages: es, registry supported languages: en

18:11:26.919 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - CreditCardRecognizer supported languages: it, registry supported languages: en

18:11:26.920 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - CreditCardRecognizer supported languages: pl, registry supported languages: en

18:11:26.922 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - EsNifRecognizer supported languages: es, registry supported languages: en

...

2026-05-12 18:11:27.091 | DEBUG    | genai_tk.workflow.anonymization.core:anonymize_text:144 - Anonymized 5 entities
2026-05-12 18:11:27.094 | SUCCESS  | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_file_task:129 - Anonymized 4 entities in support_ticket_42.txt
2026-05-12 18:11:27.100 | SUCCESS  | genai_tk.workflow.prefect.flows.anonymize_flow:anonymize_files_flow:258 - Anonymization complete: 3 processed, 0 skipped, manifest at /tmp/anon_rag_demo_pmq93tea/anonymized/anonymize_manifest.json


Processed 3 files

contract_001.txt  (hash=d29f48af...)

meeting_notes.md  (hash=f7ca7b19...)

support_ticket_42.txt  (hash=13afc488...)

### 3a. Inspect the anonymized output

Let's verify that PII has been replaced in the output files.

In [5]:
from pathlib import Path

from rich import print
from rich.columns import Columns
from rich.panel import Panel

anon_dir = Path(config.anon_dir)

for anon_file in sorted(anon_dir.glob("*.txt")) + sorted(anon_dir.glob("*.md")):
    original = (Path(config.base_dir) / anon_file.name).read_text()
    anonymized = anon_file.read_text()
    print(
        Panel(
            f"[bold]ORIGINAL:[/bold]\n{original}\n\n[bold]ANONYMIZED:[/bold]\n{anonymized}",
            title=anon_file.name,
            expand=False,
        )
    )

╭────────────────────────────────── contract_001.txt ──────────────────────────────────╮
│ ORIGINAL:                                                                            │
│ Service Agreement                                                                    │
│                                                                                      │
│ This agreement is between Alice Johnson (alice.johnson@acme.com, +1-555-123-4567)    │
│ and Bob Smith (bob.smith@globex.io, +1-555-987-6543).                                │
│                                                                                      │
│ Alice Johnson is based in New York. Bob Smith operates from London.                  │
│ Payment details: credit card 4111 1111 1111 1111, billing address: 123 Main St.      │
│                                                                                      │
│                                                                                      │
│ ANONYMIZED:                                                                          │
│ Service Agreement                                                                    │
│                                                                                      │
│ This agreement is between Allison Hill (donaldgarcia@example.net, +1-555-123-4567)   │
│ and Angie Henderson (davisjesse@example.net, +1-555-987-6543).                       │
│                                                                                      │
│ Allison Hill is based in New Jamesside. Angie Henderson operates from Robinsonshire. │
│ Payment details: credit card 4265423511615594074, billing address: 123 Main St.      │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── support_ticket_42.txt ──────────────────────────────────╮
│ ORIGINAL:                                                                                 │
│ Support Ticket #42                                                                        │
│                                                                                           │
│ Customer: Carol White                                                                     │
│ Email: carol.white@example.org                                                            │
│ Phone: (800) 555-0100                                                                     │
│ Location: San Francisco, CA                                                               │
│                                                                                           │
│ Issue: Unable to log into account. Please contact carol.white@example.org for follow-up.  │
│                                                                                           │
│                                                                                           │
│ ANONYMIZED:                                                                               │
│ Support Ticket #42                                                                        │
│                                                                                           │
│ Customer: Allison Hill                                                                    │
│ Email: donaldgarcia@example.net                                                           │
│ Phone: +1-219-560-0133                                                                    │
│ Location: Robinsonshire, CA                                                               │
│                                                                                           │
│ Issue: Unable to log into account. Please contact donaldgarcia@example.net for follow-up. │
│                                                                                           │
╰───────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── meeting_notes.md ─────────────────────────────────────────────╮
│ ORIGINAL:                                                                                                 │
│ # Q2 Planning Meeting                                                                                     │
│                                                                                                           │
│ Attendees: David Lee (david@startup.io), Emma Brown (emma.brown@corp.net)                                 │
│                                                                                                           │
│ David Lee proposed the new roadmap. Emma Brown will follow up by email at emma.brown@corp.net.            │
│ Next meeting in Chicago on 2026-06-15.                                                                    │
│                                                                                                           │
│                                                                                                           │
│ ANONYMIZED:                                                                                               │
│ # Q2 Planning Meeting                                                                                     │
│                                                                                                           │
│ Attendees: Allison Hill (donaldgarcia@example.net), Angie Henderson (davisjesse@example.net)              │
│                                                                                                           │
│ Allison Hill proposed the new roadmap. Angie Henderson will follow up by email at davisjesse@example.net. │
│ Next meeting in New Jamesside on 2026-06-15.                                                              │
│                                                                                                           │
╰───────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 3b. Direct use of `anonymize_text()` (lower-level)

The standalone helper can be called directly for single strings — useful for testing or
transforming data outside of a full Prefect flow.

In [6]:
from faker import Faker

from genai_tk.agents.langchain.middleware.presidio_detector import PresidioDetector, PresidioDetectorConfig
from genai_tk.workflow.anonymization.core import anonymize_text

detector = PresidioDetector(
    config=PresidioDetectorConfig(
        analyzed_fields=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"],
        enable_spacy=True,
    )
)
Faker.seed(42)
faker = Faker("en_US")

raw = "Please contact Jane Doe at jane.doe@secret.com or call +1-555-000-1234."
result, mapping = anonymize_text(raw, detector=detector, faker=faker)

print(f"Original : {raw}")
print(f"Anonymized: {result}")
print(f"Mapping   : {mapping}")

/home/tcl/prj/genai-tk/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


18:11:28.053 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - CreditCardRecognizer supported languages: es, registry supported languages: en

18:11:28.054 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - CreditCardRecognizer supported languages: it, registry supported languages: en

18:11:28.056 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - CreditCardRecognizer supported languages: pl, registry supported languages: en

18:11:28.057 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - EsNifRecognizer supported languages: es, registry supported languages: en

...

18:11:28.065 | WARNING | presidio-analyzer - Recognizer not added to registry because language is not supported by registry - PlPeselRecognizer supported languages: pl, registry supported languages: en

2026-05-12 18:11:28.073 | DEBUG    | genai_tk.workflow.anonymization.core:anonymize_text:144 - Anonymized 2 entities


Original : Please contact Jane Doe at jane.doe@secret.com or call +1-555-000-1234.

Anonymized: Please contact Allison Hill at donaldgarcia@example.net or call +1-555-000-1234.

Mapping   : {'Jane Doe': 'Allison Hill', 'jane.doe@secret.com': 'donaldgarcia@example.net'}

## 4. Ingest anonymized files into the vector store

We now run `rag_file_ingestion_flow` on the anonymized output directory.  
The vector store will only ever see the fake names/emails — the originals never leave the ETL step.

In [7]:
from genai_tk.workflow.prefect.flows.rag_flow import rag_file_ingestion_flow

rag_stats = run_flow_ephemeral(
    rag_file_ingestion_flow,
    base_dir=config.anon_dir,
    retriever_name=config.retriever_name,
    max_chunk_tokens=config.chunk_size,
    chunker_name=config.chunker,
    pathspecs=config.pathspecs,
    force=config.force,
    batch_size=config.batch_size,
)

print("RAG ingestion stats:", rag_stats)

2026-05-12 18:11:28.529 | INFO     | genai_tk.workflow.prefect.flows.rag_flow:rag_file_ingestion_flow:193 - Starting RAG file ingestion from '/tmp/anon_rag_demo_pmq93tea/anonymized' to retriever 'default'
2026-05-12 18:11:28.535 | INFO     | genai_tk.workflow.prefect.flows.rag_flow:rag_file_ingestion_flow:197 - Found 3 files matching patterns
2026-05-12 18:11:29.234 | DEBUG    | genai_tk.core.embeddings_store:get_vector_store:237 - Created vector store: genai_tk.core.vector_backends.ChromaBackend/embeddings_qwen3_06b => 'in-memory'
2026-05-12 18:11:29.240 | DEBUG    | genai_tk.workflow.prefect.flows.rag_flow:_prepare_files:70 - Found 0 existing file hashes in vector store
2026-05-12 18:11:29.242 | INFO     | genai_tk.workflow.prefect.flows.rag_flow:rag_file_ingestion_flow:205 - Processing 3 files, skipping 0 already processed files
2026-05-12 18:11:29.243 | INFO     | genai_tk.workflow.prefect.flows.rag_flow:rag_file_ingestion_flow:213 - Processing batch 1 (3 files)
2026-05-12 18:11:29

RAG ingestion stats:
{'total_files': 3, 'processed_files': 3, 'skipped_files': 0, 'total_chunks': 3}

## 5. Query the vector store

Retrieve chunks and confirm the anonymized text is what's stored (not the original PII).

In [8]:
from genai_tk.core.factories.retriever_factory import RetrieverFactory

managed = RetrieverFactory.create(config.retriever_name)

# Query for something we know is in the docs
results = managed.query("service agreement contact email", k=3)

print(f"Top {len(results)} results:\n")
for i, doc in enumerate(results, 1):
    print(f"[{i}] source={doc.metadata.get('source')}")
    print(f"    {doc.page_content[:200]}")
    print()

2026-05-12 18:11:31.653 | DEBUG    | genai_tk.core.embeddings_store:get_vector_store:237 - Created vector store: genai_tk.core.vector_backends.ChromaBackend/embeddings_qwen3_06b => 'in-memory'


RuntimeError: asyncio.run() cannot be called from a running event loop

## 6. YAML equivalent

The Pydantic config above maps directly to `workflows.yaml` profiles.  
The two-step `anonymize_and_ingest` workflow is already registered in `config/workflows.yaml`.

### Auto-generate the profile values from the Pydantic config

In [ ]:
import yaml

profile_values = config.model_dump()

yaml_profile = {
    "workflow_profiles": {
        "anonymize_and_ingest_docs": {
            "workflow": "anonymize_and_ingest",
            "values": profile_values,
        }
    }
}

print(yaml.dump(yaml_profile, default_flow_style=False, sort_keys=False))

### CLI equivalent

Once the profile is defined in `config/workflows.yaml` (using `${paths.data_root}` for portability),
the entire pipeline becomes a single CLI command:

```bash
# Dry-run — see what will happen
uv run cli workflow run anonymize_and_ingest_docs --dry-run

# Execute
uv run cli workflow run anonymize_and_ingest_docs

# Override source dir at runtime
uv run cli workflow run anonymize_and_ingest_docs --set base_dir=/path/to/new/docs
```

### When to use Pydantic vs YAML

| | Pydantic (notebook/script) | YAML profile (CLI/CI) |
|--|--|--|
| IDE completion | ✅ | ❌ |
| Runtime validation | ✅ | Partial |
| Programmatic transforms | ✅ | ❌ |
| Repeatable CLI invocation | ❌ | ✅ |
| Config var interpolation (`${paths.*}`) | ❌ | ✅ |
| Version-control friendly | Both | Both |

Use Pydantic during exploration; commit a YAML profile once the pipeline is stable.

## 7. Architecture summary

```
                    ┌─────────────────────────────────────────┐
                    │          anonymize_text()                │
                    │  (module-level fn in anonymization_      │
                    │   middleware.py — shared core)           │
                    └───────────────┬─────────────────────────┘
                                    │ called by
              ┌─────────────────────┴──────────────────────┐
              │                                             │
  ┌───────────▼───────────────┐           ┌────────────────▼──────────────┐
  │  anonymize_files_flow     │           │  AnonymizationMiddleware       │
  │  (ETL / batch / Prefect)  │           │  (agent runtime / LangChain)   │
  │                           │           │                                │
  │  File → anonymized file   │           │  HumanMessage → anonymized msg │
  │  + manifest.json          │           │  AIMessage    → deanonymized   │
  └───────────┬───────────────┘           └────────────────────────────────┘
              │
              ▼
  ┌───────────────────────────┐
  │  rag_file_ingestion_flow  │
  │  (chunk + embed + store)  │
  └───────────────────────────┘
```

**Key insight**: PII scrubbing can happen at two points:

1. **ETL time** (this flow) — data is anonymized *before* it enters the vector store. Safer for compliance. The RAG system never sees real PII.
2. **Agent runtime** (`AnonymizationMiddleware`) — anonymization is transparent to the user; the agent sees fake values and its responses are deanonymized before delivery.

The two strategies can be combined: anonymize at ETL, then add the middleware as a second layer for any remaining PII that may appear in user queries.